# Submission V02 · logistic regression + preprocessing (Ludovico)

**Goal:** measure on Kaggle the effect of the preprocessing chosen in `preprocessing_ludovico`.

**Same model as V01** (logistic regression); **only the preprocessing changes**, so the difference between V01 and V02 is the effect of the preprocessing alone:

| | V01 | V02 |
|---|---|---|
| NaNs | median / mode | median / mode |
| Numeric columns | standard scaler | **`log1p` + robust scaler** |
| Categoricals | one-hot, levels < 20 rows grouped | one-hot, levels **< 50** rows grouped |
| Honest CV F1 | 0.771 (Kaggle 0.770) | **expected ≈ 0.78** |

<div class="alert alert-block alert-info">

# TOC<a class="anchor" id="toc"></a>
* [<font color='#E8800A'>Setup</font>](#setup)
* [<font color='#E8800A'>1. Expected score from the validation</font>](#expected)
* [<font color='#E8800A'>2. Fit on all train and predict test</font>](#fit)
* [<font color='#E8800A'>3. Submit, publish, upload</font>](#submit)

</div>

# <font color='#E8800A'>Setup</font> <a class="anchor" id="setup"></a>
[Back to TOC](#toc)

`%run` executes `preprocessing_ludovico.ipynb` (which runs validation and cleaning). This gives back `linear_recipe`, the official folds, `evaluate`, `best_threshold` and the clean data. It takes about 2 minutes.

In [1]:
%%capture
%run ./preprocessing_ludovico.ipynb

In [2]:
from src.submissions import check_submission, publish, record_kaggle_score, submission_path, submit

model = linear_recipe(LogisticRegression(max_iter=2000))
model

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('drop', ...), ('flag', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the out

# <font color='#E8800A'>1. Expected score from the validation</font> <a class="anchor" id="expected"></a>
[Back to TOC](#toc)

Same rules as V01, on the **official folds** (3 yearly blocks + 90-day gap):
- **final threshold:** best F1 on the out-of-fold predictions of all folds;
- **honest estimate:** threshold chosen on folds 1–2, scored on fold 3. This is the number to compare with Kaggle.

In [3]:
fold_scores, oof = evaluate(model, official_folds, X, y)
covered = oof.notna()
threshold, _ = best_threshold(y[covered], oof[covered])

early = np.concatenate([va for _, va in official_folds[:-1]])
last = official_folds[-1][1]
t_early, _ = best_threshold(y.iloc[early], oof.iloc[early])
honest_f1 = f1_score(y.iloc[last], oof.iloc[last] >= t_early)

pd.Series({
    "ROC AUC (mean over folds)": fold_scores["ROC AUC"].mean(),
    "final threshold (all folds)": threshold,
    "threshold from folds 1-2": t_early,
    "honest F1: fold 3": honest_f1,
    "F1 'always 1' on fold 3 (floor)": f1_score(y.iloc[last], np.ones(len(last))),
    "V01 for comparison: honest F1 / Kaggle": "0.771 / 0.770",
}, name="value").to_frame()

,value
ROC AUC (mean over folds),0.782913
final threshold (all folds),0.38
threshold from folds 1-2,0.42
honest F1: fold 3,0.781076
F1 'always 1' on fold 3 (floor),0.753731
V01 for comparison: honest F1 / Kaggle,0.771 / 0.770


# <font color='#E8800A'>2. Fit on all train and predict test</font> <a class="anchor" id="fit"></a>
[Back to TOC](#toc)

1. Refit the pipeline on **all** clean train rows. Every step (median, log, scaler, encoder) learns on train only.
2. Predict test probabilities and apply the threshold.
3. `check_submission` verifies columns, ID order and 0/1 labels against `sample_submission.csv`.

In [4]:
final_model = clone(model).fit(X, y)

X_test = test.drop(columns=[c for c in NOT_FEATURES if c in test.columns])
test_pred = (final_model.predict_proba(X_test)[:, 1] >= threshold).astype(int)
submission = pd.DataFrame({"ID": test.index, TARGET: test_pred})
check_submission(submission)

v01 = pd.read_csv(submission_path(1))
pd.Series({
    "share of 1s - V02": submission[TARGET].mean(),
    "share of 1s - V01": v01[TARGET].mean(),
    "rows where V02 = V01": (submission[TARGET].to_numpy() == v01[TARGET].to_numpy()).mean(),
}, name="value").to_frame().style.format("{:.3f}")

,value
share of 1s - V02,0.768
share of 1s - V01,0.796
rows where V02 = V01,0.909


# <font color='#E8800A'>3. Submit, publish, upload</font> <a class="anchor" id="submit"></a>
[Back to TOC](#toc)

`submit` assigns the next free version number, saves the CSV and logs it in `docs/submissions.md`. Re-running with the same predictions reuses the version.

In [5]:
version = submit(
    submission,
    notebook="submission_v02_ludovico",
    model="logistic regression + linear recipe (median/mode, log1p, robust scaler, one-hot min 50)",
    cv_f1=honest_f1,
    notes=(f"threshold {threshold:.2f} from {OFFICIAL}; CV ROC AUC {fold_scores['ROC AUC'].mean():.3f}; "
           f"{submission[TARGET].mean():.0%} predicted 1; same model as V01, only preprocessing changed"),
)

V02 saved as submissions/ML_Group_41_Version02.csv and logged:
  Version: 02 | Date: 2026-10-15 | Author: Ludovico Piras | Notebook / model: submission_v02_ludovico · logistic regression + linear recipe (median/mode, log1p, robust scaler, one-hot min 50) | CV F1: 0.781 | Notes: threshold 0.38 from time CV + gap; CV ROC AUC 0.783; 77% predicted 1; same model as V01, only preprocessing changed
Next: publish(2) to reserve the version, then upload the file on Kaggle.


**Reserve the version on GitHub** (commit + push of the CSV and the log only), **then** upload the printed file on Kaggle:

In [ ]:
version = publish(version)

After the upload, record the public score and share it:

In [ ]:
# record_kaggle_score(version, 0.0)   # <- replace 0.0 with the public score, then run
# publish(version)

### How to read the result
- **Kaggle close to the honest estimate (≈ 0.78)** → the preprocessing gain is real and the validation still predicts Kaggle.
- **Kaggle ≈ V01 (0.770)** → the gain did not transfer. Remember that the public leaderboard is noisy (±0.02–0.03): a difference of 0.01 is within the noise in either direction.